# MiniMax H3 · ModelScope 跑档

A10 24GB · INT8 fused 21GB · 端口固定 8180 · 私有域名 minimax.gmhqyj.ccwu.cc

5 步：1 介绍 · 2 源检测+装依赖 · 3 装 ComfyUI+插件 · 4 下模型 · 5 启动+私有隧道
每步本地已有则复用；cell-2 升级 torch 后需重启内核一次。

In [ ]:
# 第 2 单元：探测 PyTorch 源速度，按需升级 torch + 装依赖
import subprocess, sys, time, urllib.request
import torch

# 探测各源速度
CU130 = [("官方", "https://download.pytorch.org/whl/cu130"),
         ("阿里云", "https://mirrors.aliyun.com/pytorch-wheels/cu130")]
best = None
for name, url in CU130:
    t0 = time.time()
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=8)
        dt = time.time() - t0
        print(name, f"{dt:.2f}s 可达")
        if best is None or dt < best[1]:
            best = (url, dt)
    except Exception as e:
        print(name, "不可达", type(e).__name__)
if not best:
    raise RuntimeError("所有 PyTorch cu130 源不可达，请检查网络")
print("最快源:", best[0])

# 其他依赖（每次装，很快）
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "modelscope", "huggingface_hub", "hf_transfer", "tqdm"])

# torch 已满足则跳过，避免重复下载 2GB+
if hasattr(torch.library, "custom_op"):
    print(f"torch {torch.__version__} 已满足，跳过 torch 安装")
else:
    print("torch 过旧，从最快源升级 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "torchaudio", "--index-url", best[0]])
    print("升级完成。⚠ 必须【重启内核/运行时】后再跑后续单元")


In [ ]:
# 第 3 单元：装 ComfyUI v0.45.3 + 插件；校验并补齐 comfy/ldm/models（本地有则复用）
import subprocess, sys, shutil, os, re
from pathlib import Path

WORKSPACE = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
ROOT = WORKSPACE / "minimax_h3"
COMFY_DIR = ROOT / "ComfyUI"
MODEL_ROOT = COMFY_DIR / "models"
COMFY_PORT = 8180
COMFY_TAG = "v0.45.3"
GIT_PROXY = "https://gh-proxy.com"
OFFICIAL = "https://github.com/Comfy-Org/ComfyUI.git"
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
remote = (GIT_PROXY.rstrip("/") + "/" if GIT_PROXY else "") + OFFICIAL

def clone(src_tag=True):
    tmp = ROOT / "_comfyui_git"
    if tmp.exists():
        shutil.rmtree(tmp)
    cmd = ["git", "clone", "--depth", "1"]
    if src_tag:
        cmd += ["--branch", COMFY_TAG]
    subprocess.check_call(cmd + [remote, str(tmp)])
    return tmp

# 1) 首次安装
if not (COMFY_DIR / "main.py").exists():
    tmp = clone()
    # 只忽略仓库根目录 models（保留 comfy/ldm/models 源码包）
    shutil.copytree(tmp, COMFY_DIR, dirs_exist_ok=True,
                    ignore=lambda s, n: {"models"} if Path(s).resolve() == tmp.resolve() else set())
    shutil.rmtree(tmp)
    print("ComfyUI 源码克隆完成")
else:
    print("ComfyUI 已存在，跳过 clone")

# 2) 关键修复：校验并补齐 comfy/ldm/models（旧版曾被 ignore_patterns 误删）
core = COMFY_DIR / "comfy" / "ldm" / "models" / "autoencoder.py"
if not core.exists():
    print("检测到 comfy/ldm/models 缺失，从官方仓库补齐 ...")
    try:
        tmp = clone(src_tag=True)
    except subprocess.CalledProcessError:
        tmp = clone(src_tag=False)
    src = tmp / "comfy" / "ldm" / "models"
    dst = COMFY_DIR / "comfy" / "ldm" / "models"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    shutil.rmtree(tmp)
print("comfy/ldm/models 校验:", core.exists())

# 3) ComfyUI 依赖（跳过 torch 等预装包）
req = COMFY_DIR / "requirements.txt"
keep = [l.strip() for l in req.read_text().splitlines()
        if l.strip() and not l.startswith("#")
        and not re.match(r"^(torch|torchvision|torchaudio|numpy|triton|xformers)", l, re.I)]
Path(ROOT / "requirements.filtered.txt").write_text("\n".join(keep) + "\n", encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(ROOT / "requirements.filtered.txt"), "cryptography>=42"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "torchsde", "einops", "transformers", "safetensors", "scipy"])
print("ComfyUI 依赖安装完成")

# 4) 自定义节点（已有则跳过）
NODES = ["city96/ComfyUI-GGUF", "Kosinkadink/ComfyUI-VideoHelperSuite",
         "matlowai/ComfyUI-MAINodes", "kijai/ComfyUI-KJNodes",
         "cubiq/ComfyUI_essentials", "ltdrdata/ComfyUI-Manager"]
NODES += ["PlagueKind/ComfyUI-PlagueKind-Nodes"]  # H3SLAAttention 稀疏注意力
nodes_root = COMFY_DIR / "custom_nodes"
nodes_root.mkdir(parents=True, exist_ok=True)
for repo in NODES:
    dst = nodes_root / repo.split("/")[-1]
    if dst.exists():
        continue
    r = GIT_PROXY.rstrip("/") + "/" if GIT_PROXY else ""
    subprocess.run(["git", "clone", "--single-branch", "--depth", "1",
                    r + "https://github.com/" + repo + ".git", str(dst)],
                   timeout=420, check=True)
print("ComfyUI + 插件安装完成，端口:", COMFY_PORT)

In [ ]:
# 第 4 单元：下载模型（本地有则复用）并软链到 ComfyUI models
from modelscope import snapshot_download
from pathlib import Path
import shutil

WORKSPACE = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
ROOT = WORKSPACE / "minimax_h3"
COMFY_DIR = ROOT / "ComfyUI"
MODEL_ROOT = COMFY_DIR / "models"
STAGING = ROOT / "_download"

RES = [
    ("MATLOWAI/minimax-h3-fused-turbo-int8-convrot",
     "diffusion_models/minimax_h3_fused_refdelta_r1024_turbo8_mystic07_int8_convrot.safetensors", "diffusion_models"),
    ("Kijai/MiniMax-H3-experimental", "minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    ("MATLOWAI/MiniMax-H3-Motion-Adapter", "temporal_expansion/minimax_h3_temporal_expansion_step375_r16.safetensors", "loras"),
]
for sub in ["diffusion_models", "vae", "text_encoders", "loras"]:
    (MODEL_ROOT / sub).mkdir(parents=True, exist_ok=True)

for repo, file, dst in RES:
    name = file.split("/")[-1]
    folder = STAGING / file.split("/")[0]
    folder.mkdir(parents=True, exist_ok=True)
    p = None
    for cand in folder.rglob(name):
        if cand.is_file():
            p = cand if p is None or cand.stat().st_size > p.stat().st_size else p
    # 阈值改为 10MB：大于即视为已下载，避免 audio_vae(0.6GB) 被误判重下
    if p is None or p.stat().st_size < 10 * 1024 * 1024:
        snapshot_download(model_id=repo, local_dir=str(folder), allow_patterns=[file])
        p = max([c for c in folder.rglob(name) if c.is_file()], key=lambda c: c.stat().st_size)
    dst_p = MODEL_ROOT / dst / name
    if not dst_p.exists() or dst_p.resolve() != p.resolve():
        if dst_p.exists() or dst_p.is_symlink():
            dst_p.unlink()
        try:
            dst_p.symlink_to(p.resolve())
        except OSError:
            shutil.copy2(p, dst_p)
    print("OK", dst + "/" + name)
print("模型下载并链接完成")

In [ ]:
# 第 5 单元：启动 ComfyUI(8180) + 私有域名隧道（Cloudflare token）
import subprocess, sys, os, time, urllib.request, re
from pathlib import Path

WORKSPACE = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
ROOT = WORKSPACE / "minimax_h3"
COMFY_DIR = ROOT / "ComfyUI"
COMFY_PORT = 8180
CF_TOKEN = "eyJhIjoiZjFhZGZmMWEwYzY5Nzg5NGZkZTExYThlNjkxM2ZhZjEiLCJ0IjoiMDdhYjZhYTAtYTJkYS00OWJkLTg1MjEtMTEwYzI5YjdkMzVlIiwicyI6Ik9UQmtaRFJpTTJFdE9ETmtaUzAwWmpBM0xXSXlZMk10T1Rrek5EaGlNams0WkdKak1HVXdNemRpTW1RdE1XWXpNUzAwWlRnekxXSTBaR0V0WlRrd05qTTBaR0poWXpabSJ9"
CF_DOMAIN = "https://minimax.gmhqyj.ccwu.cc"

def port_open(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2)
        return True
    except Exception:
        return False

# 1) 启动 ComfyUI（端口固定 8180，与 Cloudflare Public Hostname 的 ingress 对齐）
print(f"ComfyUI 端口: {COMFY_PORT}（固定，与隧道 ingress 对齐）")
if port_open(COMFY_PORT):
    print("  端口已被占用，直接复用（可能是上次没关的实例）")
else:
    log = open(ROOT / "comfyui.log", "w")
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    args = [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
            "--disable-auto-launch"]
    subprocess.Popen(args, cwd=str(COMFY_DIR), stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(90):
        if port_open(COMFY_PORT):
            break
        time.sleep(2)
comfy_ok = port_open(COMFY_PORT)
print("ComfyUI:", "OK" if comfy_ok else "失败")
if not comfy_ok:
    print((ROOT / "comfyui.log").read_text(errors="ignore")[-1500:])

# 2) cloudflared 私有隧道（token 认证）
CF_BIN = WORKSPACE / "cloudflared-linux-amd64"
if not CF_BIN.exists():
    CF_BIN = ROOT / "cloudflared"
if not CF_BIN.exists():
    print("cloudflared 二进制未找到，请先下载到", WORKSPACE / "cloudflared-linux-amd64")
else:
    os.chmod(str(CF_BIN), 0o755)
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    time.sleep(2)
    cf_log = ROOT / "cloudflared.log"
    logf = open(cf_log, "w")
    # --url 属于 tunnel 级选项，必须放在 run 之前；--token 属于 run 子命令，放在 run 之后
    cmd = [str(CF_BIN), "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}",
           "--no-autoupdate", "--protocol", "http2", "run", "--token", CF_TOKEN]
    print("启动: cloudflared tunnel --url http://127.0.0.1:%d ... run --token <token>" % COMFY_PORT)
    subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    # 等待隧道注册
    for _ in range(40):
        time.sleep(1)
        try:
            txt = cf_log.read_text(errors="ignore")
        except Exception:
            txt = ""
        if "Registered tunnel connection" in txt:
            break
        if "Incorrect tunnel secret" in txt or "Unauthorized" in txt:
            break

# 3) 输出私有域名 + 自检
print("\n" + "=" * 68)
print("  Cloudflare 私有域名:")
print("\n")
print("      " + CF_DOMAIN)
print("\n")
print("  本机调用命令:")
print(f"      python3 modelscope_minimax_generator.py ep01_s1.api.json --url {CF_DOMAIN}")
print(f"      python3 h3_check_generate.py check --url {CF_DOMAIN}")
print("\n")
print(f"  提示：域名固定不变，重启不换；ComfyUI 固定监听 127.0.0.1:{COMFY_PORT}")
print("  Dashboard → Tunnels → Public Hostname 的 Service 需为 http://localhost:8180")
for _ in range(20):
    try:
        code = urllib.request.urlopen(CF_DOMAIN + "/system_stats", timeout=10).status
    except Exception as e:
        code = getattr(e, "code", type(e).__name__)
    if code == 200:
        print("  后端自检: HTTP 200 · 私有隧道已打通 ✓")
        break
    time.sleep(3)
else:
    print(f"  后端自检: {code}（未打通，查看 {ROOT / 'cloudflared.log'}）")
print("=" * 68)
